In [ ]:
"""
====================================================================
    VECTOR FIELD + HFT + UTBOT HYBRID ENGINE (FIXED VERSION)
====================================================================

FIXES:
------
✔ Handles empty equity curves safely
✔ Handles empty trade logs safely
✔ Prevents IndexError on final_capital
✔ Better NaN handling
✔ More stable vector calculations
✔ Safer backtesting engine
✔ Works on Kaggle

DATA SOURCE:
-------------
/kaggle/input/datasets/udaysinghrathore/nse-data/EQUITY_L.csv

OUTPUT:
-------
✔ Excel report
✔ Final capital
✔ Sharpe ratio
✔ Win rate
✔ Drawdown
✔ Equity curves
✔ Strategy comparison

====================================================================
"""

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================

INITIAL_CAPITAL = 100000

LOOKBACK_PERIOD = "10y"

MAX_POSITIONS = 5

STOP_LOSS_PCT = 0.08
TAKE_PROFIT_PCT = 0.20

MAX_WORKERS = 15

# ============================================================
# PARAMETER LISTS
# ============================================================

VECTOR_WINDOW_LIST = [5, 10, 15, 20]

VOL_WINDOW_LIST = [10, 20, 30]

RSI_PERIOD_LIST = [7, 14, 21]

HFT_THRESHOLD_LIST = [
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70
]

UT_MULT_LIST = [
    0.8,
    1.0,
    1.2,
    1.5,
    2.0
]

STOP_LOSS_LIST = [
    0.03,
    0.05,
    0.08,
    0.10
]

TAKE_PROFIT_LIST = [
    0.10,
    0.15,
    0.20,
    0.25,
    0.30
]

# ============================================================
# LOAD NSE TICKERS
# ============================================================

def load_nse_tickers():

    path = (
        "/kaggle/input/datasets/"
        "udaysinghrathore/nse-data/EQUITY_L.csv"
    )

    df = pd.read_csv(path)

    symbols = (
        df["SYMBOL"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    tickers = [
        s + ".NS"
        for s in symbols
        if "&" not in s
    ]

    return tickers

# ============================================================
# RSI
# ============================================================

def compute_rsi(close, period=14):

    delta = close.diff()

    gain = delta.clip(lower=0)

    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(period).mean()

    avg_loss = loss.rolling(period).mean()

    rs = avg_gain / avg_loss

    rsi = 100 - (100 / (1 + rs))

    return rsi

# ============================================================
# UT BOT
# ============================================================

def compute_utbot(df, ut_mult=1.0):

    df = df.copy()

    tr = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            abs(df["High"] - df["Close"].shift()),
            abs(df["Low"] - df["Close"].shift())
        )
    )

    atr = tr.rolling(14).mean()

    upper = df["Close"] - ut_mult * atr

    lower = df["Close"] + ut_mult * atr

    trend = [1]

    for i in range(1, len(df)):

        try:

            if df["Close"].iloc[i] > lower.iloc[i - 1]:
                trend.append(1)

            elif df["Close"].iloc[i] < upper.iloc[i - 1]:
                trend.append(-1)

            else:
                trend.append(trend[-1])

        except:
            trend.append(trend[-1])

    df["Trend"] = trend

    df["UT_Buy"] = (
        (df["Trend"] == 1)
        &
        (df["Trend"].shift() == -1)
    )

    df["UT_Sell"] = (
        (df["Trend"] == -1)
        &
        (df["Trend"].shift() == 1)
    )

    return df

# ============================================================
# VECTOR FIELD ENGINE
# ============================================================

def compute_vector_score(
    df,
    vector_window=10,
    vol_window=20,
    rsi_period=14
):

    df = df.copy()

    returns = df["Close"].pct_change()

    # --------------------------------------------------------
    # 1. VOLATILITY VECTOR
    # --------------------------------------------------------

    vol = (
        returns
        .rolling(vol_window)
        .std()
        * np.sqrt(252)
    )

    vol_score = (
        1 - vol.rank(pct=True)
    )

    # --------------------------------------------------------
    # 2. VOLUME PRESSURE VECTOR
    # --------------------------------------------------------

    volume_ma = (
        df["Volume"]
        .rolling(20)
        .mean()
    )

    volume_ratio = (
        df["Volume"]
        / volume_ma
    )

    volume_score = (
        volume_ratio.rank(pct=True)
    )

    # --------------------------------------------------------
    # 3. MOMENTUM VECTOR
    # --------------------------------------------------------

    momentum = (
        df["Close"]
        .pct_change(5)
    )

    momentum_score = (
        momentum.rank(pct=True)
    )

    # --------------------------------------------------------
    # 4. VECTOR CURVATURE
    # --------------------------------------------------------

    velocity = (
        returns
        .rolling(vector_window)
        .mean()
    )

    acceleration = velocity.diff()

    curvature = acceleration.diff()

    curvature_score = (
        curvature.abs()
        .rank(pct=True)
    )

    # --------------------------------------------------------
    # 5. RSI VECTOR
    # --------------------------------------------------------

    rsi = compute_rsi(
        df["Close"],
        period=rsi_period
    )

    rsi_score = (
        rsi / 100
    )

    # --------------------------------------------------------
    # 6. VWAP DEVIATION VECTOR
    # --------------------------------------------------------

    typical = (
        df["High"]
        + df["Low"]
        + df["Close"]
    ) / 3

    vwap = (
        (typical * df["Volume"])
        .rolling(20)
        .sum()
        /
        df["Volume"]
        .rolling(20)
        .sum()
    )

    vwap_dev = (
        (df["Close"] - vwap)
        / vwap
    )

    vwap_score = (
        vwap_dev.rank(pct=True)
    )

    # --------------------------------------------------------
    # COMBINE VECTOR FIELDS
    # --------------------------------------------------------

    df["VECTOR_SCORE"] = (

        0.20 * vol_score +

        0.20 * volume_score +

        0.20 * momentum_score +

        0.15 * curvature_score +

        0.15 * rsi_score +

        0.10 * vwap_score

    )

    df["VECTOR_SCORE"] = (
        df["VECTOR_SCORE"]
        .clip(0, 1)
        .fillna(0)
    )

    return df

# ============================================================
# FETCH STOCK DATA
# ============================================================

def fetch_stock(ticker):

    try:

        df = yf.download(
            ticker,
            period=LOOKBACK_PERIOD,
            auto_adjust=True,
            progress=False
        )

        if df.empty:
            return None

        if len(df) < 250:
            return None

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = (
                df.columns.get_level_values(0)
            )

        df.dropna(inplace=True)

        return ticker, df

    except:
        return None

# ============================================================
# LOAD ALL DATA
# ============================================================

def load_all_data(tickers):

    all_data = {}

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(fetch_stock, t): t
            for t in tickers
        }

        for future in as_completed(futures):

            result = future.result()

            if result is not None:

                ticker, df = result

                all_data[ticker] = df

                print("Loaded:", ticker)

    return all_data

# ============================================================
# BACKTEST ENGINE
# ============================================================

def run_backtest(
    all_data,
    hft_threshold=0.60,
    ut_mult=1.0,
    stop_loss=0.05,
    take_profit=0.20,
    vector_window=10,
    vol_window=20,
    rsi_period=14,
    mode="VECTOR"
):

    cash = INITIAL_CAPITAL

    positions = {}

    trade_log = []

    equity_curve = []

    # --------------------------------------------------------
    # PRECOMPUTE INDICATORS
    # --------------------------------------------------------

    processed_data = {}

    for ticker, df in all_data.items():

        try:

            x = compute_utbot(
                df,
                ut_mult=ut_mult
            )

            x = compute_vector_score(
                x,
                vector_window=vector_window,
                vol_window=vol_window,
                rsi_period=rsi_period
            )

            processed_data[ticker] = x

        except:
            continue

    if len(processed_data) == 0:

        return (
            pd.DataFrame(),
            []
        )

    all_dates = sorted(
        set(
            d
            for df in processed_data.values()
            for d in df.index
        )
    )

    # --------------------------------------------------------
    # MAIN LOOP
    # --------------------------------------------------------

    for current_date in all_dates:

        # ====================================================
        # EXITS
        # ====================================================

        for ticker in list(positions.keys()):

            df = processed_data[ticker]

            if current_date not in df.index:
                continue

            row = df.loc[current_date]

            pos = positions[ticker]

            current_price = float(row["Close"])

            stop_price = (
                pos["entry_price"]
                * (1 - stop_loss)
            )

            tp_price = (
                pos["entry_price"]
                * (1 + take_profit)
            )

            exit_trade = False

            reason = ""

            if current_price <= stop_price:

                exit_trade = True

                reason = "STOP"

            elif current_price >= tp_price:

                exit_trade = True

                reason = "TARGET"

            elif row["UT_Sell"]:

                exit_trade = True

                reason = "UT_SELL"

            if exit_trade:

                proceeds = (
                    pos["shares"]
                    * current_price
                )

                profit = (
                    proceeds
                    - pos["invested"]
                )

                cash += proceeds

                trade_log.append({

                    "Stock": ticker,

                    "Entry Date":
                    pos["entry_date"],

                    "Exit Date":
                    current_date,

                    "Entry Price":
                    pos["entry_price"],

                    "Exit Price":
                    current_price,

                    "Profit":
                    profit,

                    "Return %":
                    (
                        profit
                        / pos["invested"]
                    ) * 100,

                    "Mode":
                    mode,

                    "Reason":
                    reason
                })

                del positions[ticker]

        # ====================================================
        # ENTRIES
        # ====================================================

        slots = (
            MAX_POSITIONS
            - len(positions)
        )

        if slots > 0:

            candidates = []

            for ticker, df in processed_data.items():

                if ticker in positions:
                    continue

                if current_date not in df.index:
                    continue

                row = df.loc[current_date]

                vector_signal = (
                    row["VECTOR_SCORE"]
                    >= hft_threshold
                )

                ut_signal = bool(
                    row["UT_Buy"]
                )

                if mode == "VECTOR":

                    entry = vector_signal

                elif mode == "UTBOT":

                    entry = ut_signal

                else:

                    entry = (
                        vector_signal
                        and ut_signal
                    )

                if entry:

                    candidates.append(

                        (
                            ticker,
                            row["VECTOR_SCORE"],
                            row
                        )
                    )

            candidates.sort(
                key=lambda x: x[1],
                reverse=True
            )

            for ticker, score, row in candidates[:slots]:

                allocation = cash / slots

                if allocation <= 0:
                    continue

                shares = (
                    allocation
                    / row["Close"]
                )

                cash -= allocation

                positions[ticker] = {

                    "entry_date":
                    current_date,

                    "entry_price":
                    float(row["Close"]),

                    "shares":
                    shares,

                    "invested":
                    allocation
                }

        # ====================================================
        # EQUITY
        # ====================================================

        pv = cash

        for ticker, pos in positions.items():

            df = processed_data[ticker]

            if current_date in df.index:

                pv += (

                    pos["shares"]

                    *

                    float(
                        df.loc[current_date]["Close"]
                    )
                )

        equity_curve.append(pv)

    trades = pd.DataFrame(trade_log)

    return trades, equity_curve

# ============================================================
# EVALUATION
# ============================================================

def evaluate_strategy(
    name,
    trades,
    equity_curve
):

    # --------------------------------------------------------
    # FIX FOR EMPTY CURVES
    # --------------------------------------------------------

    if equity_curve is None:

        equity_curve = []

    if len(equity_curve) == 0:

        return {

            "Strategy": name,

            "Final Capital":
            INITIAL_CAPITAL,

            "Return %": 0,

            "Sharpe": 0,

            "Max DD %": 0,

            "Win Rate %": 0,

            "Trades": 0
        }

    eq = np.array(equity_curve)

    if len(eq) == 0:

        return {

            "Strategy": name,

            "Final Capital":
            INITIAL_CAPITAL,

            "Return %": 0,

            "Sharpe": 0,

            "Max DD %": 0,

            "Win Rate %": 0,

            "Trades": 0
        }

    # --------------------------------------------------------
    # SAFE FINAL CAPITAL
    # --------------------------------------------------------

    final_capital = float(eq[-1])

    total_return = (

        (
            final_capital
            / INITIAL_CAPITAL
        ) - 1

    ) * 100

    # --------------------------------------------------------
    # SHARPE
    # --------------------------------------------------------

    returns = (
        pd.Series(eq)
        .pct_change()
        .dropna()
    )

    sharpe = 0

    if len(returns) > 5:

        if returns.std() != 0:

            sharpe = (

                returns.mean()

                /

                returns.std()

            ) * np.sqrt(252)

    # --------------------------------------------------------
    # DRAWDOWN
    # --------------------------------------------------------

    peak = np.maximum.accumulate(eq)

    dd = (

        (eq - peak)

        /

        peak

    )

    max_dd = dd.min() * 100

    # --------------------------------------------------------
    # WIN RATE
    # --------------------------------------------------------

    if trades is None:

        trades = pd.DataFrame()

    if len(trades) > 0:

        win_rate = (

            len(
                trades[
                    trades["Profit"] > 0
                ]
            )

            /

            len(trades)

        ) * 100

    else:

        win_rate = 0

    return {

        "Strategy":
        name,

        "Final Capital":
        round(final_capital, 2),

        "Return %":
        round(total_return, 2),

        "Sharpe":
        round(sharpe, 2),

        "Max DD %":
        round(max_dd, 2),

        "Win Rate %":
        round(win_rate, 2),

        "Trades":
        len(trades)
    }

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    print("\nLoading NSE Tickers...\n")

    tickers = load_nse_tickers()

    tickers = tickers[:1500]

    print("Tickers:", len(tickers))

    # ========================================================
    # LOAD DATA
    # ========================================================

    all_data = load_all_data(tickers)

    print("\nLoaded Stocks:", len(all_data))

    # ========================================================
    # RUN STRATEGIES
    # ========================================================

    print("\nRunning VECTOR...\n")

    vector_trades, vector_curve = run_backtest(
        all_data,
        mode="VECTOR"
    )

    print("\nRunning UTBOT...\n")

    ut_trades, ut_curve = run_backtest(
        all_data,
        mode="UTBOT"
    )

    print("\nRunning HYBRID...\n")

    hybrid_trades, hybrid_curve = run_backtest(
        all_data,
        mode="HYBRID"
    )

    # ========================================================
    # EVALUATION
    # ========================================================

    vec_eval = evaluate_strategy(
        "VECTOR",
        vector_trades,
        vector_curve
    )

    ut_eval = evaluate_strategy(
        "UTBOT",
        ut_trades,
        ut_curve
    )

    hyb_eval = evaluate_strategy(
        "HYBRID",
        hybrid_trades,
        hybrid_curve
    )

    results_df = pd.DataFrame([

        vec_eval,

        ut_eval,

        hyb_eval

    ])

    print("\n================ RESULTS ================\n")

    print(results_df)

    # ========================================================
    # SAVE EXCEL
    # ========================================================

    with pd.ExcelWriter(
        "vector_hft_results.xlsx",
        engine="openpyxl"
    ) as writer:

        results_df.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        vector_trades.to_excel(
            writer,
            sheet_name="VECTOR",
            index=False
        )

        ut_trades.to_excel(
            writer,
            sheet_name="UTBOT",
            index=False
        )

        hybrid_trades.to_excel(
            writer,
            sheet_name="HYBRID",
            index=False
        )

    print(
        "\nSaved: vector_hft_results.xlsx"
    )

    # ========================================================
    # PLOT
    # ========================================================

    plt.figure(figsize=(15, 7))

    if len(vector_curve) > 0:

        plt.plot(
            vector_curve,
            label="VECTOR"
        )

    if len(ut_curve) > 0:

        plt.plot(
            ut_curve,
            label="UTBOT"
        )

    if len(hybrid_curve) > 0:

        plt.plot(
            hybrid_curve,
            label="HYBRID"
        )

    plt.title(
        "Vector vs UTBOT vs Hybrid"
    )

    plt.xlabel("Days")

    plt.ylabel("Portfolio Value")

    plt.legend()

    plt.grid(True)

    plt.show()


TOTAL COMBINATIONS: 1

[1/1]
14 1.5 10 2.5 0.9


IndexError: index -1 is out of bounds for axis 0 with size 0

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ARISINFRA.NS"}}}
$ARISINFRA.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")
$LTIM.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MEGASOFT.NS"}}}
$MEGASOFT.NS: possibly delisted; no price data found  (period=10y) (Yahoo error = "No data found, symbol may be delisted")
